In [ ]:
%pip install numpy matplotlib qiskit qiskit-aer scipy --quiet

In [ ]:
from __future__ import annotations
import math
from dataclasses import dataclass
from fractions import Fraction
from typing import Callable, Sequence
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT, MCXGate
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator
import layer2
import layer4
import layer5
import BarrenPlateaus_HC-GO

In [ ]:
@dataclass
class ApplicationTestConfig:  #Shared structure
    seed:int = 7
    shots:int = 4096
    aer_backend:AerSimulator | None = None

    def simulator(self) -> AerSimulator:
        return self.aer_backend or AerSimulator()

def heralded_retry(inner_circuit:QuantumCircuit) -> QuantumCircuit:
    op = QuantumCircuit(*inner_circuit.qregs)
    op.compose(inner_circuit, inplace=True)
    return op

In [ ]:
#Cost models

def grover_iters(addr:int, n:int):
    N = 2**addr
    if n<=0:
        return 0
    return int(math.pi/4.0*math.sqrt(N/n))

def ShorCostModel(rsa_bits:int, window_size:int):
    toffoli_count = 3*rsa_bits**3
    logical_qubits = 2*rsa_bits+3
    physical_qubits = int(logical_qubits*2000)
    qrom_size = 2**window_size
    return rsa_bits, physical_qubits, qrom_size, toffoli_count

In [ ]:
#Test 1: Grover search with a simulated QRAM-loaded db

def _grover_diffusion(addr:int) -> QuantumCircuit:
    qr = QuantumRegister(addr, name="addr")
    qc = QuantumCircuit(qr, name="FroverDiffusion")

    for q in qr:
        qc.h(q)
        qc.x(q)
    qc.h(qr[-1])

    if addr > 1:
        qc.append(MCXGate(addr-1), list(qr))
    else:
        qc.z(qr[0])
    qc.h(qr[-1])

    for q in qr:
        qc.x(q)
        qc.h(q)

    return qc

def  _qram_grover_sim(db:Sequence[int], addr:int) -> QuantumCircuit:
    qram_query = fanout(db)
    addr_reg = qram_query.qregs[0]
    diffusion = _grover_diffusion(addr)
    iter = QuantumCircuit(*qram_query.qregs, name=f"GroverIter(n={addr})")
    iter.compose(qram_query, inplace=True)
    iter.compose(diffusion, qubits=addr_reg, inplace=True)

    return iter

def  grover_e2e(addr:int, marked:int | None = None, cfg:ApplicationTestConfig | None = None) -> dict[str, object]:
    rng = np.random.default_rng(cfg.seed)
    N = 2**addr
    if marked is None:
        marked = int(rng.integers(0, N))

    db = [0]*N    #Simulated data from the SiC memory write op, in form of a classical db
    db[marked] = 1

    qram_query = fanout(db)
    addr_reg = qram_query.qregs[0]
    data_reg = qram_query.qregs[1]

    k_opt = grover_iters(addr,marked=1)
    theta = math.asin(math.sqrt(1.0/N))
    p_th = math.sin((2.0*k_opt+1.0)*theta)**2

    cbits = ClassicalRegister(addr)
    pipeline = QuantumCircuit(*qram_query.qregs, cbits, name="GroverPipeline")

    for q in addr_reg:    #Initial state of uniform superposition on address register
        pipeline.h(q)
    pipeline.x(data_reg[0])
    pipeline.h(data_reg[0])

    iter_block = _qram_grover_sim(db, addr)    #Optimal k grover iterations with heralded retries
    iter_wrap = heralded_retry(iter_block)
    for _ in range(k_opt):
        pipeline.compose(iter_wrap, inplace=True)

    pipeline.measure(addr_reg, cbits)

    counts = cfg.simulator().run(pipeline, shots=cfg.shots, seed_simulator=cfg.seed).result().get_counts()

    marked_bitstring = format(marked, f"0{addr}b")
    marked_measure = counts.get(marked_bitstring, 0)
    measured_prob = marked_measure/cfg.shots
    amp_factor = measured_prob*N
    top_address = int(max(counts, key=counts.get))

    return {
        "addr": addr,
        "db_size": N,
        "marked_index": marked,
        "db_preview": db[:8] + (["..."] if N > 8 else []),
        "grover_iterations": k_opt,
        "pipeline_qubit_count": pipeline.num_qubits,
        "pipeline_circuit_depth": pipeline.depth(),
        "theoretical_success_prob":  p_th,
        "measured_success_prob": measured_prob,
        "baseline_prob": 1.0/N,
        "amplification_factor": amp_factor,
        "top_measured_address": top_address,
        "match": top_address == marked,
        "pass": (measured_prob > 0.5) and if match,
      }


In [ ]:
#Test 2: Shor's factoring with QRAM-backed windowed modular exponentiation

def _modular_mult_table(a:int, N:int, count:int) -> list[int]:   #Pre-compute a mod N and store in QRAM which will be accessed later
    return [pow(a, 2**k, N) for k in range(count)]

def _shor_qram_lookup(a:int, N:int, count:int) -> QuantumCircuit:
    table = _modular_mult_table(a, N, count)
    data = 4    #Each product value is converted into its 4-bit representation, with its own fanout circuit
    qram_segments = []

    for bit_idx in range(data):
        bit_column = [(val >> bit_idx) & 1 for val in table]
        padded_size = 2**int(math.ceil(math.log2(len(bit_column))))
        bit_column_padded = list(bit_column) + [0] *(padded_size - len(bit_column))
        segment = fanout(bit_column_padded)
        qram_segments.append(segment)

    segment_addr_size = qram_segments[0].qregs[0].size
    addr = QuantumRegister(segment_addr_size, name="addr")
    data = QuantumRegister(data, name="data")
    composite = QuantumCircuit(addr, data, name="ShorQRAMTable")

    for bit_idx, segment in enumerate(qram_segments):   #Map the individual segments onto a common circuit
        segment_qubits = list(addr) + [data[bit_idx]]
        composite.compose(segment, qubits=segment_qubits, inplace=True)

    return composite

def _shor_phase_est(a:int, N:int, count:int) -> QuantumCircuit:
    count_reg = QuantumRegister(count)
    work_reg = QuantumRegister(4)
    cbits = ClassicalRegister(count)
    qc = QuantumCircuit(count_reg, work_reg, cbits)

    _ = _shor_qram_lookup(a, N, count)  #Combine the lookup table from QRAM

    for q in count:
        qc.h(q)
    qc.x(work[0])

    for k in range(count):  #Demo
        product = pow(a, 2**k, N)
        if product == 1:
            continue
        if product == 7:
            qc.cswap(count[k], work[2], work[3])
            qc.cswap(count[k], work[1], work[2])
            qc.cswap(count[k], work[0], work[1])
        elif product == 4:
            qc.cswap(count[k], work[1], work[3])
            qc.cswap(count[k], work[0], work[2])
        else:
            raise NotImplementedError(f"Controlled multiplication by {product} mod {N} not implemented for this test")

    iqft = QFT(count, inverse=True, do_swaps=True).decompose()    #Inverse QFT on the counting register
    qc.compose(iqft, qubits=count, inplace=True)
    qc.measure(count, cbits)

    return qc

def shor_e2e(a:int, N:int, count:int, cfg:ApplicationTestConfig | None = None) -> dict[str, object]:
    qram_segment = _shor_qram_lookup(a, N, count)
    qc = _shor_phase_est(a, N, count)
    retry = heralded_retry(qc)
    counts = cfg.simulator().run(retry, shots=cfg.shots, seed_simulator=cfg.seed).result().get_counts()

    sorted_phases = sorted(((int(bits, 2), i) for bits, i in counts.items()), key=lambda t: -t[1])
    top_outcomes = sorted_phases[:4]

    factors: tuple[int, ...] = ()
    order = None
    for phase, _freq in sorted_phases:
        if phase == 0:
            continue
        for max_denom in range(N-1, 1, -1):
            frac = Fraction(phase, 2**count).limit_denominator(max_denom)
            other_choice = frac.denominator
            if other_choice <= 1 or other_choice%2 != 0:
                continue
            if pow(a, other_choice, N) != 1:
                continue
            half_power = pow(a, other_choice//2, N)
            if half_power == N-1:
                continue
            f1 = math.gcd(half_power-1, N)
            f2 = math.gcd(half_power+1, N)
            choice = tuple(sorted({f1, f2} - {1, N}))
            if choice and math.prod(choice) == N:
                factors = choice
                order = other_choice
                break
        if factors:
            break

    return {
        "N": N,
        "a": a,
        "count": count,
        "qram_table": _modular_mult_table(a, N, count),
        "qram_segment_qubit_count": qram_segment.num_qubits,
        "qram_segment_depth": qram_segment.depth(),
        "measured_phase_top4": top_outcomes,
        "inferred_order": order,
        "factors": list(factors),
        "pass": N == math.prod(factors) if factors else False,
    }

In [ ]:
#Test 3: HCGO mitigated QAOA with QRAM loaded edge weights

def _edge_weights(edges:Sequence[tuple[int, int]], weights:Sequence[float]) -> QuantumCircuit:
    size = len(edges)
    addr = max(1, int(math.ceil(math.log2(size))))
    padded_size = 2**addr
    bit_column = [1 if w != 0 else 0 for w in weights]
    bit_column += [0]*(padded_size-size)
    return fanout(bit_column)


def _cost_hamil(edges:Sequence[tuple[int, int]], qubits:int) -> SparsePauliOp:   #MaxCut cost Hamiltonian
    terms = []
    for i, j in edges:
        pauli_zz = ["I"]*qubits
        pauli_zz[i] = "Z"
        pauli_zz[j] = "Z"
        terms.append(("".join(reversed(pauli_zz)), -0.5))
        terms.append(("I"*qubits, 0.5))
    return SparsePauliOp.from_list(terms)

def hcgo_e2e(block_qubits:int, qcr_qubits:int, sizes:Tuple, circ_depth:int, samples:int, iters:int, cfg:ApplicationTestConfig | None = None)-> dict[str, object]:
    res = []
    seg_info = None

    for s in sizes:   #Building the graph
        edges = [(i, (i+1)%s) for i in range(s)]
        weights = [1.0]*len(edges)
        qram_tile = _edge_weights(edges, weights)
        if seg_info is None:
            seg_info = {"qubit_count": qram_tile.num_qubits, "depth": qram_tile.depth(), "edges": len(edges)}

        hamiltonian = _cost_hamil(edges, s)
        params = s*circ_depth
        var_hea = _estimate_gradient_variance(    ##
            build_ansatz=lambda p: _hea_ansatz(s, circ_depth, p),
            params=params,
            observable=hamiltonian,
            samples=samples,
            rng=rng,
        )

        blocks = s//block_qubits
        hcgo_hamil = [generate_local_hamiltonian(block_qubits) for _ in range(blocks)]  ##
        _trained_params, training_gradient_norms = run_hcgo_training(
            global_hamiltonian_terms=hcgo_hamil,
            num_blocks=blocks,
            block_size=block_qubits,
            context_size=qcr_qubits,
            iterations=iters,
        )

        gradients_per_block = list(map(list, zip(*training_gradient_norms)))
        block_var = [float(np.var(norms)) for norms in gradients_per_block]
        var_hcgo = float(np.mean(block_var))

        res.append({
            "qubits": s,
            "edges": len(edges),
            "blocks": blocks,
            "hea_variance": var_hea,
            "hcgo_variance": var_hcgo,
            "per_block_variance": block_var,
            "mitigation_ratio": (var_hcgo / var_hea if var_hea > 0 else float("inf")),
        })

    hcgo_vars = [r["hcgo_variance"] for r in res]   #Comparing variance scaling in mitigated vs un-mitigated
    hea_vars = [r["hea_variance"] for r in res]
    hcgo_spread = max(hcgo_vars) / max(min(hcgo_vars), 1e-15)
    hea_spread = max(hea_vars) / max(min(hea_vars), 1e-15)

    return {
        "block_qubits": block_qubits,
        "qcr_qubits": qcr_qubits,
        "qram_tile": seg_info,
        "per_size": res,
        "hea_variance_spread": hea_spread,
        "hcgo_variance_spread": hcgo_spread,
        "pass": hcgo_spread < hea_spread,
    }

def _plot_hcgo(result:dict[str, object]) -> None:   #Plotting the final comparison
    sizes = [r["qubits"] for r in result["per_size"]]
    hea_vars = [r["hea_variance"] for r in result["per_size"]]
    hcgo_vars = [r["hcgo_variance"] for r in result["per_size"]]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    x = np.arange(len(sizes))
    width = 0.35
    axes[0].bar(x - width/2, hea_vars, width, label="HEA (unmitigated)", color="blue")
    axes[0].bar(x + width/2, hcgo_vars, width, label="HC-GO (mitigated)", color="red")
    axes[0].set_yscale("log")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"N = {n}" for n in sizes])
    axes[0].set_ylabel("Gradient variance (log scale)")
    axes[0].set_title("Gradient variance: HEA vs HC-GO across system sizes")
    axes[0].legend()
    axes[0].grid(True, axis="y", alpha=0.3, which="both")
    axes[1].semilogy(sizes, hea_vars, "o-", color="blue", label="HEA (unmitigated)", linewidth=2, markersize=8)
    axes[1].semilogy(sizes, hcgo_vars, "s-", color="#red", label="HC-GO (mitigated)", linewidth=2, markersize=8)
    axes[1].set_xlabel("System size N (qubits)")
    axes[1].set_ylabel("Gradient variance (log scale)")
    axes[1].set_title("Gradient variance scaling with system size")
    axes[1].set_xticks(sizes)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, which="both")

    plt.tight_layout()
    plt.show()

In [ ]:
#Executable

def _applications_tests() -> None:
    print("Layer 6: End-to-end application tests")

    config = ApplicationTestConfig(seed=7, shots=4096)

    #Grover e2e
    print("Grover demo:")
    for i in (3, 4):
        grover = grover_e2e(addr=i, cfg=config)
        print(f"N = {grover['addr']} qubits ({grover['db_size']} entries)")
        print(f"Marked index = {grover['marked_index']}")
        print(f"DB preview = {grover['db_preview']}")
        print(f"Pipeline qubit count = {grover['pipeline_qubit_count']}")
        print(f"Pipeline circuit depth = {grover['pipeline_circuit_depth']}")
        print(f"Grover iterations k* = {grover['grover_iterations']}")
        print(f"Theoretical P(marked) = {grover['theoretical_success_prob']:.4f}")
        print(f"Measured P(marked) = {grover['measured_success_prob']:.4f}")
        print(f"Amplification factor = {grover['amplification_factor']:.2f}x")
        print(f"Top measured address = {grover['top_measured_address']}")
        print(f"Matches = {grover['match']}")
        print(f"Result = {'PASS' if grover['pass'] else 'FAIL'}")

    #Shor e2e
    print("Shor Demo")
    for base in (7,):
        shor = shor_e2e(a=base, N=15, count=4, cfg=config)
        print(f"Factoring N = 15 at base a = {base}")
        print(f"QRAM table = {shor['qram_table']}")
        print(f"QRAM segment qubit count = {shor['qram_segment_qubit_count']}")
        print(f"QRAM segment depth = {shor['qram_segment_depth']}")
        print(f"Top-4 phase outcomes = {shor['measured_phase_top4']}")
        print(f"Inferred order = {shor['inferred_order']}")
        print(f"Extracted factors = {shor['factors']}")
        print(f"Result = {'PASS' if shor['pass'] else 'FAIL'}")

    print("Shor cost model projections at RSA scale:")
    for s in (1024, 2048, 3072):
        cost = ShorCostModel(s, window_size=5)
        print(f"RSA-{cost.rsa_bits}:")
        print(f"{cost.logical_qubits:>5} logical qubits")
        print(f"{cost.toffoli_count:.2e} Toffolis")
        print(f"~{cost.physical_qubits_surface_code:.2e} physical qubits")
        print(f"QROM size {cost.windowed_arithmetic_qrom_size}")

    #HCGO e2e
    print("HCGO Demo")
    hcgo = hcgo_e2e(block_qubits=2, qcr_qubits=2, sizes=(4, 6, 8), circ_depth=8, samples=20, iters=3, cfg=config)
    _plot_hcgo(hcgo)
    print(f"Result = {'PASS' if hcgo['pass'] else 'FAIL'}")

In [ ]:
if __name__ == "__main__":
    _applications_tests()